## 1. Kiểm tra môi trường Kaggle và GPU

In [ ]:
import os
import sys
import platform
import subprocess
from pathlib import Path

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())
print("Kaggle:", bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")))

try:
    import torch
    import torchvision
    print("PyTorch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("CUDA version:", torch.version.cuda)
    else:
        print("WARNING: GPU chưa được bật. Vào Settings → Accelerator → GPU.")
except Exception as exc:
    print("Không kiểm tra được PyTorch:", repr(exc))

if Path("/kaggle/input").exists():
    print("\nTop-level /kaggle/input:")
    for p in sorted(Path("/kaggle/input").iterdir()):
        print(" -", p)


## 2. Clone repository GitHub

Sửa `REPO_URL` thành URL repository của bạn. Repository public có thể clone trực tiếp. Với repository private, hãy cấu hình xác thực an toàn trong Kaggle Secrets; không ghi token trực tiếp vào notebook.

Nếu source code đã được đính kèm như một Kaggle Input, đặt `USE_GIT = False` và `PROJECT_DIR` thành đường dẫn thư mục source code đó.


In [ ]:
# TODO: thay bằng URL GitHub repository của bạn
REPO_URL = "https://github.com/YOUR_USERNAME/fairface-gender-classification.git"

# Nếu source code đã được thêm qua Kaggle Input:
USE_GIT = True
PROJECT_DIR = Path("/kaggle/working/fairface-gender-classification")

if USE_GIT:
    if "YOUR_USERNAME" in REPO_URL or REPO_URL.strip() == "":
        raise ValueError(
            "Hãy sửa REPO_URL ở cell này thành URL GitHub thật của project."
        )
    if PROJECT_DIR.exists():
        print("Repository directory đã tồn tại:", PROJECT_DIR)
    else:
        subprocess.run(
            ["git", "clone", REPO_URL, str(PROJECT_DIR)],
            check=True
        )
else:
    # TODO: thay bằng đường dẫn source code trong /kaggle/input
    PROJECT_DIR = Path("/kaggle/input/YOUR-SOURCE-CODE-DATASET")
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(f"Không tìm thấy source code: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
print("Project:", PROJECT_DIR)
print("Files:", [p.name for p in PROJECT_DIR.iterdir()][:40])

required = [
    PROJECT_DIR / "configs" / "default.yaml",
    PROJECT_DIR / "scripts" / "prepare_splits.py",
    PROJECT_DIR / "src" / "train.py",
    PROJECT_DIR / "src" / "evaluate.py",
    PROJECT_DIR / "src" / "compare.py",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Repository thiếu file cần thiết:\n" + "\n".join(missing) +
        "\nHãy kiểm tra URL/branch và cấu trúc repository."
    )


## 3. Cài dependencies

Kaggle thường đã có PyTorch, torchvision, NumPy, pandas, scikit-learn, matplotlib, seaborn, PyYAML, tqdm, SciPy và Pillow. Cell này chỉ cài các gói trong `requirements.txt` nếu file đó tồn tại; nếu việc cài đặt làm thay đổi PyTorch/torchvision và gây lỗi CUDA, hãy kiểm tra lại `requirements.txt` để tránh pin phiên bản không tương thích.


In [ ]:
os.chdir(PROJECT_DIR)
req = PROJECT_DIR / "requirements.txt"

if req.exists():
    print("Installing requirements from", req)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )
else:
    print("Không có requirements.txt; bỏ qua bước pip install.")

# Recheck after installation
import torch, torchvision, pandas as pd, numpy as np
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())


## 4. Tìm dataset FairFace trong Kaggle Inputs

Thêm dataset bằng nút **Add Input**. Cell này tìm các CSV tên phổ biến bên trong `/kaggle/input`, đọc cột `file` để kiểm tra đường dẫn ảnh và in ra các ứng viên. Nếu có nhiều dataset cùng chứa CSV, bạn nên chọn đúng thư mục chứa bản **padding 0.25** bằng cách điền `DATA_DIR_OVERRIDE`.

Không có bước nào tự tải một dataset slug không được chỉ định.


In [ ]:
from pathlib import Path
import pandas as pd

INPUT_ROOT = Path("/kaggle/input")
CSV_NAMES = {
    "train_labels.csv",
    "val_labels.csv",
    "fairface_label_train.csv",
    "fairface_label_val.csv",
}

csv_candidates = []
if INPUT_ROOT.exists():
    for p in INPUT_ROOT.rglob("*"):
        if p.is_file() and p.name.lower() in CSV_NAMES:
            csv_candidates.append(p)

print(f"Found {len(csv_candidates)} candidate label CSV file(s):")
for p in csv_candidates[:100]:
    print(" -", p)

# Nếu có nhiều nguồn, điền đường dẫn dataset root tại đây.
# Ví dụ: Path("/kaggle/input/my-fairface-dataset")
DATA_DIR_OVERRIDE = None

def infer_dataset_root(csv_path: Path) -> Path:
    # CSV thường nằm tại root dataset; có thể nằm sâu hơn nên dò vài cấp.
    candidates = [csv_path.parent, *list(csv_path.parents)[1:4]]
    for root in candidates:
        names = {p.name.lower() for p in root.iterdir() if p.is_file()} if root.exists() else set()
        has_train = bool(names & {"train_labels.csv", "fairface_label_train.csv"})
        has_val = bool(names & {"val_labels.csv", "fairface_label_val.csv"})
        if has_train and has_val:
            return root
    return csv_path.parent

if DATA_DIR_OVERRIDE is not None:
    DATA_DIR = Path(DATA_DIR_OVERRIDE)
    if not DATA_DIR.exists():
        raise FileNotFoundError(f"DATA_DIR_OVERRIDE không tồn tại: {DATA_DIR}")
elif csv_candidates:
    roots = []
    for p in csv_candidates:
        r = infer_dataset_root(p)
        if r not in roots:
            roots.append(r)
    print("\nDataset root candidates:")
    for r in roots:
        print(" -", r)
    if len(roots) == 1:
        DATA_DIR = roots[0]
    else:
        raise RuntimeError(
            "Tìm thấy nhiều thư mục dữ liệu. Hãy đặt DATA_DIR_OVERRIDE thành "
            "đường dẫn đúng của dataset padding 0.25 rồi chạy lại cell."
        )
else:
    raise FileNotFoundError(
        "Không tìm thấy CSV nhãn FairFace trong /kaggle/input. "
        "Hãy bấm Add Input và thêm dataset chứa train/validation CSV cùng ảnh."
    )

print("\nSelected DATA_DIR:", DATA_DIR)
print("Top-level entries:")
for p in list(DATA_DIR.iterdir())[:40]:
    print(" -", p.name)


## 5. Kiểm tra CSV và ảnh trước khi chia tập

Cell này chỉ kiểm tra cấu trúc và một số đường dẫn mẫu. Nếu cột hoặc đường dẫn khác kỳ vọng, hãy sửa `DATA_DIR`/cấu trúc dataset thay vì bỏ qua lỗi.


In [ ]:
def find_label_csv(root: Path, kind: str):
    if kind == "train":
        names = {"train_labels.csv", "fairface_label_train.csv"}
    else:
        names = {"val_labels.csv", "fairface_label_val.csv"}
    matches = [p for p in root.rglob("*") if p.is_file() and p.name.lower() in names]
    if not matches:
        raise FileNotFoundError(f"Không tìm thấy CSV {kind} trong {root}")
    if len(matches) > 1:
        print(f"WARNING: có nhiều CSV {kind}; dùng file đầu tiên:", matches[0])
    return matches[0]

train_csv = find_label_csv(DATA_DIR, "train")
val_csv = find_label_csv(DATA_DIR, "val")
train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)

print("Train CSV:", train_csv, train_df.shape)
print("Validation CSV:", val_csv, val_df.shape)
print("Train columns:", train_df.columns.tolist())
print("Validation columns:", val_df.columns.tolist())

required_cols = {"file", "gender", "race", "age"}
for name, df in [("train", train_df), ("validation", val_df)]:
    missing_cols = required_cols - set(df.columns)
    if missing_cols:
        raise ValueError(f"{name} CSV thiếu cột: {sorted(missing_cols)}")
    print(f"\n{name} gender distribution:")
    print(df["gender"].value_counts(dropna=False))
    print(f"{name} race distribution:")
    print(df["race"].value_counts(dropna=False))

print("\nTrain file examples:")
print(train_df["file"].head(5).to_list())


## 6. Chọn thư mục output và chuẩn bị splits

Kaggle Input là read-only. Tất cả split, log và checkpoint phải ghi vào `/kaggle/working`.

`prepare_splits.py` theo hướng dẫn của project nhận `--data_dir`; nếu script có hỗ trợ `--out`, notebook truyền `splits` để lưu trong project working copy.


In [ ]:
OUTPUT_DIR = Path("/kaggle/working/outputs")
SPLITS_DIR = PROJECT_DIR / "splits"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

prepare_script = PROJECT_DIR / "scripts" / "prepare_splits.py"
help_result = subprocess.run(
    [sys.executable, str(prepare_script), "--help"],
    capture_output=True, text=True
)
print(help_result.stdout)
if help_result.returncode != 0:
    print(help_result.stderr)

prepare_cmd = [
    sys.executable, str(prepare_script),
    "--data_dir", str(DATA_DIR),
]
# Chỉ thêm --out nếu script khai báo tùy chọn đó.
if "--out" in help_result.stdout:
    prepare_cmd += ["--out", str(SPLITS_DIR)]

print("Running:", " ".join(prepare_cmd))
subprocess.run(prepare_cmd, check=True)

print("\nSplit files:")
for p in SPLITS_DIR.glob("*"):
    print(" -", p, f"({p.stat().st_size:,} bytes)")


## 7. Chọn mô hình: M0 — ResNet50 baseline

Notebook này chỉ huấn luyện **resnet50_base**. Muốn chạy kiến trúc khác, hãy mở notebook tương ứng.


In [ ]:
# Model được cố định cho notebook này
MODEL_NAME = "resnet50_base"
EPOCHS = 25
SEED = 42

print("Model:", MODEL_NAME)
print("Epochs:", EPOCHS)
print("Seed:", SEED)
print("Output:", OUTPUT_DIR / MODEL_NAME / str(SEED))


## 8. Smoke test (khuyến nghị)

Chạy thử pipeline trước khi train đủ epoch. Nếu source code không hỗ trợ `--debug`, hãy xem `python -m src.train --help` và sửa lệnh theo CLI thực tế.


In [ ]:
os.chdir(PROJECT_DIR)

train_help = subprocess.run(
    [sys.executable, "-m", "src.train", "--help"],
    capture_output=True, text=True
)
print(train_help.stdout)
if train_help.returncode != 0:
    print(train_help.stderr)

if "--debug" in train_help.stdout:
    cmd = [
        sys.executable, "-m", "src.train",
        "--model", MODEL_NAME,
        "--config", "configs/default.yaml",
        "--seed", str(SEED),
        "--debug",
        "--data_dir", str(DATA_DIR),
        "--output_dir", str(OUTPUT_DIR),
    ]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    print("CLI không công bố --debug; bỏ qua smoke test tự động.")


## 9. Huấn luyện mô hình đã chọn

Bạn có thể đổi `EPOCHS` ở cell cấu hình phía trên. Mặc định là 25 epoch theo tài liệu đặc tả. `--resume` tiếp tục từ checkpoint nếu code của project hỗ trợ đúng cơ chế này.


In [ ]:
os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, "-m", "src.train",
    "--model", MODEL_NAME,
    "--config", "configs/default.yaml",
    "--seed", str(SEED),
    "--epochs", str(EPOCHS),
    "--resume",
    "--data_dir", str(DATA_DIR),
    "--output_dir", str(OUTPUT_DIR),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(
        "Training thất bại. Kiểm tra CLI help của project; "
        "nếu --data_dir/--output_dir không được hỗ trợ, cấu hình đường dẫn trong YAML."
    )


## 10. Đánh giá mô hình

Chỉ đánh giá test sau khi đã hoàn tất chọn checkpoint bằng validation.


In [ ]:
os.chdir(PROJECT_DIR)

cmd = [
    sys.executable, "-m", "src.evaluate",
    "--model", MODEL_NAME,
    "--seed", str(SEED),
    "--data_dir", str(DATA_DIR),
    "--output_dir", str(OUTPUT_DIR),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(
        "Evaluation thất bại. Chạy `python -m src.evaluate --help` và điều chỉnh tham số CLI."
    )


## 11. Đóng gói output để tải xuống

In [ ]:
from pathlib import Path
import zipfile

print("Output directory:", OUTPUT_DIR)
if OUTPUT_DIR.exists():
    for p in sorted(OUTPUT_DIR.rglob("*")):
        if p.is_file():
            print(f"{p} ({p.stat().st_size / (1024**2):.2f} MB)")

ZIP_PATH = Path("/kaggle/working/resnet50_base_outputs.zip")
if OUTPUT_DIR.exists():
    with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        model_root = OUTPUT_DIR / MODEL_NAME
        if model_root.exists():
            for p in model_root.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=p.relative_to(OUTPUT_DIR))
        else:
            for p in OUTPUT_DIR.rglob("*"):
                if p.is_file():
                    zf.write(p, arcname=p.relative_to(OUTPUT_DIR))
    print("ZIP created:", ZIP_PATH, f"({ZIP_PATH.stat().st_size / (1024**2):.2f} MB)")
